# Unit-Aware Parameters with `Param()`

Demonstrates `Param()`, `to_example()`, `with_units()`, and automatic pint Quantity stripping using a single comprehensive model.

In [1]:
from meshly import Packable, Param, InlineArray, Array
from pint import UnitRegistry
import numpy as np

ureg = UnitRegistry()

class Simulation(Packable):
    """Covers all field types: float, InlineArray, Array, nested Packable."""
    velocity: InlineArray = Param(units="m/s", example=[30.0, 0, 0], shape=(3,), description="Inlet velocity")
    temperature: float = Param(300.0, units="K", description="Fluid temperature")
    pressure: float = Param(101325.0, units="Pa", description="Static pressure")
    displacements: Array = Param(units="m", example=np.zeros((4, 3), dtype=np.float32), shape=(None, 3), description="Nodal displacements")
    label: str = Param("default", units="dimensionless", description="Run label")

In [2]:
import json

# Units appear in the JSON schema (this is what MCP tools / UIs see)
schema = Simulation.model_json_schema()
for name in ["velocity", "temperature", "pressure", "displacements"]:
    props = schema["properties"][name]
    print(f"{name}: units={props.get('units')}")

velocity: units=m/s
temperature: units=K
pressure: units=Pa
displacements: units=m


In [3]:
# to_example() builds an instance from Param example/default values
sim = Simulation.to_example()
print(f"velocity:      {sim.velocity}")
print(f"temperature:   {sim.temperature}")
print(f"displacements: {sim.displacements.shape}")

velocity:      [30.  0.  0.]
temperature:   300.0
displacements: (4, 3)


In [4]:
# with_units() converts fields to pint Quantities
su = sim.with_units()
print(f"velocity:    {su.velocity}")
print(f"to km/h:     {su.velocity.to('km/h')}")
print(f"temperature: {su.temperature}")
print(f"to Celsius:  {su.temperature.to('degC')}")
print(f"pressure:    {su.pressure.to('atm')}")
print(f"displace mm: {su.displacements.to('mm')}")

velocity:    [30.0 0.0 0.0] meter / second
to km/h:     [108.0 0.0 0.0] kilometer / hour
temperature: 300.0 kelvin
to Celsius:  26.850000000000023 degree_Celsius
pressure:    1.0 standard_atmosphere
displace mm: [[0.0 0.0 0.0] [0.0 0.0 0.0] [0.0 0.0 0.0] [0.0 0.0 0.0]] millimeter


In [5]:
# Pass pint Quantities directly — auto-converted to declared units, stored as plain values
sim2 = Simulation(
    velocity=ureg.Quantity([108.0, 0, 0], "km/h"),
    temperature=ureg.Quantity(26.85, "degC"),
    pressure=ureg.Quantity(1.0, "atm"),
    displacements=ureg.Quantity(np.array([[1, 0, 0]], dtype=np.float32), "mm"),
)
print(f"velocity (m/s):      {sim2.velocity} ({type(sim2.velocity).__name__})")
print(f"temperature (K):     {sim2.temperature} ({type(sim2.temperature).__name__})")
print(f"pressure (Pa):       {sim2.pressure} ({type(sim2.pressure).__name__})")
print(f"displacements (m):   {sim2.displacements} ({type(sim2.displacements).__name__})")

velocity (m/s):      [30.  0.  0.] (ndarray)
temperature (K):     300.0 (float)
pressure (Pa):       101325.0 (float)
displacements (m):   [[0.001 0.    0.   ]] (ndarray)


In [6]:
# Error: passing units to a field without Param raises TypeError
class NoUnits(Packable):
    value: float = 0.0

try:
    NoUnits(value=ureg.Quantity(42.0, "m/s"))
except TypeError as e:
    print(f"Expected error: {e}")

# Dimensionless quantities on non-Param fields are fine
m = NoUnits(value=ureg.Quantity(42.0, "dimensionless"))
print(f"dimensionless OK: {m.value}")

Expected error: Field 'value' on NoUnits received a pint Quantity with units 'meter / second', but no Param(units=...) is defined for this field. Either add Param(units=...) or pass a plain value.
dimensionless OK: 42.0
